In [9]:
from data import CleanNumericDatasets
from reservoirs import CPRC
from circuits import CPCircuit
from ELM import QuantumELM
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score
)
from utility import Evaluator
from visualize import plot_all_scores_per_model, add_elm_to_results, print_key_scores, plot_scores_heatmap
from benchmark import run_classical_benchmarks

In [2]:
dim = 20
ds = CleanNumericDatasets(seed=3, scale=True)

X_train, X_test, y_train, y_test = ds.get(
    name="openml:covertype",
    sample_size=2000,
    feature_size=dim,
    return_split=True,
    test_size=0.2
)

ds.info(print_out=True)

/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/datasets/_openml.py:328: UserWarning: Multiple active versions of the dataset matching the name covertype exist. Versions may be fundamentally different, returning version 3. Available versions:
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=150
- version 1, status: active
  url: https://www.openml.org/search?type=data&id=180

  warn(warning_msg)


name: openml:covertype (covertype)
task: classification
source: openml
n_samples_requested: 2000
n_features_requested: 20
n_samples_returned: 2000
n_features_returned: 20
n_classes: 7
feature_transform: selectkbest
scaler: standard
split: train_test_split
test_size: 0.2
random_state: 3
stratified: True
y_type: int64
X_dtype: float32
notes: OpenML dataset. Enforced numeric X; labels encoded if needed. Transforms/scaling fit on train only.


/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [20 27 28 38 41 50] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


{'name': 'openml:covertype (covertype)',
 'task': 'classification',
 'source': 'openml',
 'n_samples_requested': 2000,
 'n_features_requested': 20,
 'n_samples_returned': 2000,
 'n_features_returned': 20,
 'n_classes': 7,
 'feature_transform': 'selectkbest',
 'scaler': 'standard',
 'split': 'train_test_split',
 'test_size': 0.2,
 'random_state': 3,
 'stratified': True,
 'y_type': 'int64',
 'X_dtype': 'float32',
 'notes': 'OpenML dataset. Enforced numeric X; labels encoded if needed. Transforms/scaling fit on train only.'}

In [ ]:
CP_params = [-np.pi/3, np.pi/6, -np.pi/9, np.pi/7, np.pi/9, -np.pi/7]
cprc = CPRC(dim=dim, execution_mode='simulation', CP_params=CP_params, kernel = True)

In [13]:
elm = QuantumELM(
        reservoir=cprc, 
        regularization=10, 
        show_progress=True, 
        model_type='ridge_classifier',  # Options: 'ridge', 'lasso', 'linear', 'svr', 'svc',  or "ridge_classifier","logreg", "svc" for classification
        )
elm.fit(X_train, y_train)

ELM feature extraction: 100%|██████████| 1600/1600 [01:22<00:00, 19.33 sample/s]


In [14]:
predictions = elm.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print("RMSE:", rmse)

ELM feature extraction: 100%|██████████| 400/400 [00:22<00:00, 17.96 sample/s]

RMSE: 1.404457190518814


In [16]:
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    classification_report, 
    confusion_matrix
)

# 1. Summary Report (Best for quick human-readable overview)
# This includes precision, recall, and f1-score for every class
print("--- Classification Report ---")
print(classification_report(y_test, predictions))

# 2. Confusion Matrix (To see where the model is getting confused)
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, predictions))

# 3. Individual Programmatic Metrics
# Use 'macro' for balanced class importance or 'weighted' for real-world distribution
metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision (Macro)": precision_score(y_test, predictions, average='macro', zero_division=0),
    "Recall (Macro)": recall_score(y_test, predictions, average='macro', zero_division=0),
    "F1 (Macro)": f1_score(y_test, predictions, average='macro', zero_division=0),
}

print("\n--- Key Scores ---")
for name, score in metrics.items():
    print(f"{name}: {score:.4f}")


--- Classification Report ---
              precision    recall  f1-score   support

           0       0.66      0.69      0.67       144
           1       0.72      0.76      0.74       194
           2       0.60      0.52      0.56        23
           3       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         9
           5       0.36      0.31      0.33        13
           6       0.62      0.50      0.55        16

    accuracy                           0.68       400
   macro avg       0.42      0.40      0.41       400
weighted avg       0.66      0.68      0.66       400

--- Confusion Matrix ---
[[ 99  41   0   0   1   0   3]
 [ 43 147   2   0   0   1   1]
 [  1   4  12   0   0   6   0]
 [  0   0   0   0   0   0   1]
 [  1   8   0   0   0   0   0]
 [  0   3   6   0   0   4   0]
 [  7   1   0   0   0   0   8]]

--- Key Scores ---
Accuracy: 0.6750
Precision (Macro): 0.4222
Recall (Macro): 0.3964
F1 (Macro): 0.4076


/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [17]:
balanced_accuracy_score(y_test, predictions)

0.39638048526999675

In [7]:
results = run_classical_benchmarks(X_train, y_train, X_test, y_test, seed=3, cv_splits=5)
for k, v in results.items():
    print("\n", k)
    print(" best_params:", v["best_params"])
    print(" cv_best_balanced_acc:", v["cv_best_balanced_acc"])
    print(" test_metrics:", v["test_metrics"])

Benchmark models:   0%|          | 0/6 [00:00<?, ?it/s]/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/utkarshsingh/miniforge3/envs/qml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Us


 logreg
 best_params: {'clf__C': 100}
 cv_best_balanced_acc: 0.4762714901073057
 test_metrics: {'accuracy': 0.71, 'balanced_accuracy': 0.41866211568699246, 'f1': 0.4429654189199511, 'precision': 0.49505007399744244, 'recall': 0.41866211568699246, 'roc_auc_ovr_macro': 0.9145863538304624}

 svm_rbf
 best_params: {'clf__C': 10, 'clf__gamma': 0.01}
 cv_best_balanced_acc: 0.4804865735450334
 test_metrics: {'accuracy': 0.72, 'balanced_accuracy': 0.40822678989420363, 'f1': 0.4048269968544883, 'precision': 0.42190727675378653, 'recall': 0.40822678989420363, 'roc_auc_ovr_macro': 0.9123516402654258}

 random_forest
 best_params: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1}
 cv_best_balanced_acc: 0.44796281708845553
 test_metrics: {'accuracy': 0.755, 'balanced_accuracy': 0.4481557121348695, 'f1': 0.4767073075690663, 'precision': 0.5322047282748534, 'recall': 0.4481557121348695, 'roc_auc_ovr_macro': 0.9297391133181875}

 grad_boost
 best_params: {'learning_rate': 0.03, 'max_de